# Weight Tying Analysis

Analyze the embedding-unembedding relationship: alignment measurement, subspace analysis, norm distributions, isotropy, and token neighborhood consistency.

## Why This Matters

Weight tying examines the structure and statistics of model parameters. The structure of weights constrains what computations a component can perform, and spectral analysis can reveal functional specialization, redundancy, and low-rank structure.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_tying_analysis import (
    embedding_unembed_alignment, embedding_subspace_analysis,
    norm_distribution, embedding_isotropy, token_neighborhood_analysis,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print('Model ready')

In [ ]:
result = embedding_unembed_alignment(model)
print(f'Mean cosine: {result["mean_cosine_similarity"]:.4f}')
print(f'Approximately tied: {result["is_approximately_tied"]}')
print(f'Most aligned: {result["most_aligned"][:3]}')

In [ ]:
result = embedding_subspace_analysis(model)
print(f'Embed rank: {result["embedding_effective_rank"]:.2f}')
print(f'Unembed rank: {result["unembed_effective_rank"]:.2f}')
print(f'Subspace overlap: {result["subspace_overlap"]:.4f}')
print(f'Shared dimensions: {result["shared_dimensions"]}')

In [ ]:
result = norm_distribution(model)
print(f'Norm correlation: {result["norm_correlation"]:.4f}')
print(f'Mean embed norm: {result["mean_embed_norm"]:.4f}')
print(f'Mean unembed norm: {result["mean_unembed_norm"]:.4f}')

In [ ]:
result = embedding_isotropy(model)
print(f'Isotropy score: {result["isotropy_score"]:.4f}')
print(f'Mean cosine: {result["mean_cosine"]:.4f}')
print(f'Principal direction dominance: {result["principal_direction_dominance"]:.4f}')

In [ ]:
result = token_neighborhood_analysis(model, token_ids=[0, 5, 10, 15, 20])
print(f'Mean consistency: {result["mean_consistency"]:.4f}')
for t in result['per_token']:
    print(f'  Token {t["token_id"]}: consistency={t["neighborhood_consistency"]:.3f}')